# 3 · A clinical quality measure, end to end
A published **example measure** — a folder of `.cql` files — compiled for Spark and run over FHIR resources,
showing the per-patient results and how to ask follow-up questions of the same helper library the measure uses.

**Nothing is authored in this notebook.** The measure is not a string in a cell; it is a folder of
`.cql` files, and it runs **by name**. That is the whole point of this module: a measure is
content you publish and version, not code you paste. No CQL on Databricks App, no registry, no UI — the
compiler, a folder, and Spark.

The example measure is a hypertension control measure with an initial population, exclusions and a numerator,
over 4,000 synthetic patients, about 1,000 synthesized for each example measure. In a real measure everything is
larger but the shape is the same.

**Runs on serverless — no cluster to attach.** Just **Run All**.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

## 1. The content beside this notebook
Two directories sit next to this notebook, holding everything one measure needs, kept apart from
the notebook that runs it:

```
measures/     the measure and the eligibility library it includes
data/         FHIR NDJSON, one file per resource type
```

In a real deployment the measures folder is a **Databricks Git Folder** — your measures repo, synced by the
platform — or a Unity Catalog Volume. It is an ordinary directory, which is why no
platform-specific code appears anywhere below.

The paths are derived from this notebook's own location, so the same cell works in the repo, in a
bundle copy, and in `/Shared/CqlDatabricks/examples` with nothing baked in.

**Expected output:** the resolved folder, then the two content directories — for example
```
content : /Workspace/Shared/CqlDatabricks/examples
found   : ['data', 'measures']
```

In [0]:
import pathlib

# The notebook's own workspace folder, under the /Workspace mount that makes workspace files ordinary
# files — the content directories are its siblings.
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
CONTENT = pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.relative_to("/")

# Fail loud with the path rather than later with a confusing empty result.
missing = [d for d in ("measures", "data") if not (CONTENT / d).is_dir()]
if missing:
    raise FileNotFoundError(
        f"{missing} not found beside this notebook in {CONTENT} — they are published alongside it; if you "
        f"copied the notebook on its own, copy them too, or point CONTENT at your own content directory")

print("content :", CONTENT)
print("found   :", sorted(d for d in ("data", "measures")))

## 2. The data: the default environment
Every run is in one environment: a Unity Catalog catalog whose `data` schema holds one view per FHIR type,
named for the type (`condition`), each returning the resource as a `resource` struct and `patient_id`, the
key of the patient the row belongs to. The deploy builds the default environment with the patients in
`data/` beside this notebook, so there is nothing to load here. The engine lowers every `[Resource]` retrieve
to a read of its view, and the plan filters above it.

The read schema is **the compiler's, not inferred**: each resource has the struct the compiler's FHIR model
defines (`client.resource_schema("Condition")` shows it). In your own environment the views select your
governed FHIR tables; the CQL does not change.

**Expected output:** the environment's name and catalog, then one line per type: its row count and the view
it is read from.

In [0]:
from cql_databricks.resolvers import EnvironmentResolver
from cql_databricks.data_bindings import spark_runner

environment = EnvironmentResolver(spark_runner(spark)).default()
print("environment:", environment.name, "in catalog", environment.catalog)
for t in ("Condition", "Encounter", "Patient"):
    view = f"{environment.schema('data')}.{t.lower()}"
    print(f"{t:12} {spark.table(view).count()} rows -> {view}")

### Is the data readable?
Before running a measure over it, one `%%cql` cell asks the data a direct question: self-contained
CQL, no dependencies. The cell declares the question, and the Python cell after it runs it.

**Expected output:** 4,000 rows, one per patient, each with true or false for whether they have any condition
on record — a direct check to verify the data is readable.

In [0]:
%%cql
library MountCheck version '1.0.0'
using FHIR version '4.0.1'

context Patient

define "Has A Condition": exists ([Condition])

In [0]:
display(cql(library="MountCheck").exec("Has A Condition"))

## 3. Mount the libraries — a folder of CQL as the library source
This is the piece that makes a *published* measure runnable rather than a pasted one.

The example measure `include`s one helper library, and the compile service resolves **none** of them
— by design. It is a dumb endpoint: the caller resolves the `include` closure and ships every
dependency; the service only merges what it is handed. That purity boundary
is why the service reads no filesystem, no registry and no network — and why the resolver lives here,
next to the files.

`library_path` mounts one or more directories as the session's library source. Roots are **ordered
and earlier wins**, so an override directory shadows a vendored content repo:

```python
connect(spark, library_path=["/Workspace/Users/me/wip", "/Volumes/acme/content/measures"])
```

A client this notebook does not build itself — the `%%cql` magic builds its own — reads the same
folder from `CQL_DATABRICKS_LIBRARY_PATH` or `spark.cqldatabricks.libraryPath` instead.

**Expected output:**
```
libraries at /Workspace/…/examples/measures
  FHIRHelpers|4.0.1
  HypertensionBPControl|1.0.0
  MeasureCommon|1.0.0
```

In [0]:
from cql_databricks.compiler import connect

MEASURES = str(CONTENT / "measures")

client = connect(spark, library_path=MEASURES)
print("libraries at", MEASURES)
for ref in client.libraries("*"):
    print(" ", ref)

## 4. Bind the measure by name, and read it before running it
`client.cql(library="HypertensionBPControl")` addresses the measure by **name**. Resolving it walks
the `include` closure, so the common libraries and helper functions are found and shipped with the
compile without being mentioned here.

Two rules decide which source an `include` resolves to:

- **Sibling-first.** An `include` resolves against the root's own directory before the wider search
  path — a content repo can ship a copy of its shared libraries *per measure*, so a flat scan sees
  several different copies of the same library.
- **Exact version.** An `include` that states a version resolves only to that version; a version
  nothing on the search path declares fails loud, naming the versions it did find.

The handle is lazy and the compile is memoized, so inspecting and then running costs one round trip.
No data has been touched at this point.

**Expected output:**
```
populations : ['Initial Population', 'Denominator', 'Denominator Exclusions', 'Numerator']
defines     : [... 14 defines including population definitions, hypertension logic and blood pressure
           criteria ...]
```
`populations` finds the eCQM populations by their well-known names; it is what `exec()` runs when you name
no targets. The measure has four populations: initial population, denominator, exclusions and numerator.
Over the 2026-01-01 to 2026-12-31 period, 590 of 4,000 patients are in the initial population.

In [0]:
measure = client.cql(library="HypertensionBPControl") \
                .with_measurement_period("2026-01-01", "2026-12-31")

print("populations :", measure.populations())
print("defines     :", measure.defines())

## 5. Run it
`exec()` with the population defines evaluates them together, compiling the shared graph
**once** — the measures' logic flows through each other, and that shared work is computed once.

Every define's result arrives with **identical schemas** (`__subject_id`, `__val`), because a define names the
object and never a column. The frames are presented side by side for each patient so you can
see the pattern: which patients have hypertension, which are in scope, which are excluded, and which meet control.

**Expected output:** 4,000 rows, one per patient. 590 are in the initial population (adults 18–85 with
hypertension and a qualifying visit). Of those, 543 are not excluded (not in hospice or palliative care, not
pregnant, no kidney failure). Of those 543, 337 have controlled blood pressure — a rate of about 62%.

In [0]:
from pyspark.sql import functions as F

from cql_databricks.compiler import VALUE_COLUMN

defines = measure.exec("Initial Population", "Denominator", "Denominator Exclusions", "Numerator")
# Name each result for the SQL cell in section 6. A named library's run reads its views under a prefix of its own.
for name, df in defines.items():
    df.createOrReplaceTempView(name.replace(" ", "_"))
report = (defines["Initial Population"].withColumnRenamed(VALUE_COLUMN, "initial_population")
          .join(defines["Denominator"].withColumnRenamed(VALUE_COLUMN, "denominator"), "__subject_id")
          .join(defines["Denominator Exclusions"].withColumnRenamed(VALUE_COLUMN, "excluded"), "__subject_id")
          .join(defines["Numerator"].withColumnRenamed(VALUE_COLUMN, "numerator"), "__subject_id")
          .orderBy("__subject_id"))
report.display()

## 6. Query the result as SQL
The cell above named each define's frame as a session temp view, with spaces as underscores
(`Initial_Population`, `Denominator`, etc.), so the headline rate is one SQL cell away — no CQL, no Python, nothing to export.

**Expected output:** one row — the rate of controlled blood pressure among the denominator (after exclusions).

Each define is true or false on its own, so the query nests them the way CQF Measures scores a proportion: the
numerator counts only patients in the denominator and not excluded.

In [0]:
%sql
SELECT count_if(d.__val) AS denominator,
       count_if(d.__val AND NOT coalesce(ex.__val, false)) AS denominator_after_exclusions,
       count_if(d.__val AND NOT coalesce(ex.__val, false) AND n.__val) AS numerator
FROM Denominator d
LEFT JOIN Denominator_Exclusions ex ON d.__subject_id = ex.__subject_id
LEFT JOIN Numerator n ON d.__subject_id = n.__subject_id

## 7. Ask a question the measure did not answer
A care-gap report always provokes a follow-up: *which* patients, and why are they excluded? The published
definitions in the helper libraries are addressable, so the follow-up can be asked of **the same libraries the measure uses**
rather than re-implemented in ad-hoc SQL — which is where consistency usually breaks, someone
rebuilding "age at end of period" slightly differently and getting a number nobody can reconcile.

The follow-up here asks the helper library the measure includes, `MeasureCommon`, one of the
questions behind its exclusions: who was in hospice care during the period?

**Where an `include` is found.** A `%%cql` cell's `include` resolves against earlier cells, then the
folder in `spark.cqldatabricks.libraryPath`, then the environment's knowledge. This
notebook hands its folder to `connect()` only, so the magic's own client cannot see it; the follow-up
therefore runs the helper library by name with `library=`. Set the conf to the folder and a
`%%cql` cell can `include` it too.

**Expected output:** 4,000 rows, one per patient. 42 are true: they had hospice care during 2026, or were
discharged to hospice in it.

In [0]:
followup = client.cql(library="MeasureCommon").with_measurement_period("2026-01-01", "2026-12-31")
followup.exec("In Hospice Care").orderBy("__subject_id").display()

## 8. The same measure, a different period
The measurement period is a `parameter`, and the handle is immutable-fluent — `with_measurement_period`
returns a NEW handle rather than rebinding this one. Nothing about the content changes; the question
is asked of a different year.

Watch the population turn over. For 2026 the measure reported 11 patients in the initial population; for 2025
there are 0 — every hypertension patient's qualifying visit in this dataset is in 2026.

That is worth sitting with: **a rate is meaningless without its period**, and the same content
honestly produces both numbers.

Each run of a named library reads its views under a prefix of its own, so this run leaves the
frames from section 5, and the views section 6 read, on 2026. One session, one library, two
parameterizations, and neither run disturbs the other.

**Expected output:**
```
2025: initial population = 0
```

In [0]:
prior = measure.with_measurement_period("2025-01-01", "2025-12-31").exec("Initial Population")
initial_pop_2025 = prior.collect()
count_2025 = sum(1 for r in initial_pop_2025 if r[VALUE_COLUMN])
print(f"2025: initial population = {count_2025}")

---
### What just happened
A measure held as content in a folder, its dependencies resolved by name, its data read through one mount
contract — and a per-patient result, a SQL count, an ad-hoc follow-up asking the helper library the same
question, and the same measure re-parameterized to a different year, all producing honest numbers that
cannot disagree with each other. No application was involved.

**04 · Compilation and inspection** takes a measure (different content) and never runs it at all: the define
graph, the ELM, and the Spark plan the compiler produces before any data is touched.

In [0]:
# Every cell above ran, so the notebook is green. There are no value assertions here by design: the
# narrative states each expected result, and a compile or runtime failure fails its own cell loudly
# rather than being collected into a summary at the end.
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "03_clinical_measure"}))